# 04 — Construcción y preprocesamiento de features

**Input:** `data/processed/species_features.parquet`  
**Outputs:**
- `data/processed/aves_train.parquet` — aves con IUCN (para entrenamiento)
- `data/processed/aves_predict.parquet` — aves sin IUCN (para predicción)
- `data/processed/mamiferos_train.parquet` — mamíferos con IUCN
- `data/processed/mamiferos_predict.parquet` — mamíferos sin IUCN

**Pasos:**
1. Inspección rápida
2. Split por clase
3. Definición de features por clase
4. Imputación de nulos
5. Encoding de categóricas
6. Split train / predict
7. Guardado

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../data/processed')

df = pd.read_parquet(DATA / 'species_features.parquet', engine='fastparquet').reset_index()
print(f'Shape: {df.shape}')
df.head(3)

Shape: (1704, 52)


,species,class,iucn_categoria,n_occ,lat_centroid,lon_centroid,year_min,year_max,bio1,bio4,...,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle,Range.Size,body_mass_g,trophic_level,activity_cycle,range_area_km2,threatened
0,None,Mammalia,None,1,-17.168056,-69.395278,2003,2003,5.125573,208.983109,...,NaN,None,None,None,NaN,24.91,NaN,3.0,722551.83,NaN
1,None,Mammalia,LC,2,-16.236531,-68.191953,2017,2024,5.151948,160.726135,...,NaN,None,None,None,NaN,34.50,2.0,2.0,506394.71,0.0
2,None,Aves,LC,2,-14.474833,-68.558166,2000,2000,20.647385,117.548706,...,2.0,Carnivore,Vertivore,Generalist,14309701.27,NaN,NaN,NaN,NaN,0.0


In [34]:
# Fix: valores nodata del raster que no se enmascararon (e.g. -3.4e+38 como float32)
# Cualquier valor bio fuera de rango real → NaN, luego se imputan con la mediana del train

BIO_COLS = ['bio1', 'bio4', 'bio7', 'bio12', 'bio14', 'bio15']

# Umbrales conservadores por variable WorldClim
BIO_VALID = {
    'bio1':  (-60,   40),    # temperatura media anual (°C)
    'bio4':  (0,    20000),  # estacionalidad temperatura (°C*100)
    'bio7':  (0,     700),   # rango temperatura anual (°C*10)
    'bio12': (0,   12000),   # precipitación anual (mm)
    'bio14': (0,    1000),   # precipitación mes más seco (mm)
    'bio15': (0,     300),   # estacionalidad precipitación (CV)
}

for col, (lo, hi) in BIO_VALID.items():
    n_bad = ((df[col] < lo) | (df[col] > hi)).sum()
    df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
    print(f'{col}: {n_bad} valores nodata → NaN')

print(f'\nNulos bio tras limpieza:')
print(df[BIO_COLS].isnull().sum())

bio1: 4 valores nodata → NaN
bio4: 4 valores nodata → NaN
bio7: 4 valores nodata → NaN
bio12: 4 valores nodata → NaN
bio14: 4 valores nodata → NaN
bio15: 4 valores nodata → NaN

Nulos bio tras limpieza:
bio1     6
bio4     6
bio7     6
bio12    6
bio14    6
bio15    6
dtype: int64


## 1. Inspección rápida

In [35]:
print('=== Distribución por clase ===')
print(df['class'].value_counts())

print('\n=== Categorías IUCN ===')
print(df['iucn_categoria'].value_counts(dropna=False))

print('\n=== Target threatened ===')
print(df['threatened'].value_counts(dropna=False))

print('\n=== Nulos por columna (solo las que tienen) ===')
nulls = df.isnull().sum()
print(nulls[nulls > 0])

=== Distribución por clase ===
class
Aves        1478
Mammalia     226
Name: count, dtype: int64

=== Categorías IUCN ===
iucn_categoria
LC      1392
None     202
NT        61
VU        45
CR         4
Name: count, dtype: int64

=== Target threatened ===
threatened
0.0    1453
NaN     202
1.0      49
Name: count, dtype: int64

=== Nulos por columna (solo las que tienen) ===
species               1704
iucn_categoria         202
bio1                     6
bio4                     6
bio7                     6
bio12                    6
bio14                    6
bio15                    6
Beak.Length_Culmen     387
Beak.Width             387
Beak.Depth             387
Tarsus.Length          387
Wing.Length            387
Kipps.Distance         387
Mass                   387
Habitat                390
Habitat.Density        387
Migration              388
Trophic.Level          387
Trophic.Niche          387
Primary.Lifestyle      387
Range.Size             387
body_mass_g           1533
tr

In [36]:
# Verificar que threatened está bien construido
check = df.dropna(subset=['threatened'])
print('threatened=1 (amenazadas):', int(check['threatened'].sum()))
print('threatened=0 (no amenazadas):', int((check['threatened'] == 0).sum()))
print('sin IUCN (a predecir):', df['threatened'].isna().sum())

print('\nCategorías de las amenazadas:')
print(df[df['threatened'] == 1]['iucn_categoria'].value_counts())

threatened=1 (amenazadas): 49
threatened=0 (no amenazadas): 1453
sin IUCN (a predecir): 202

Categorías de las amenazadas:
iucn_categoria
VU    45
CR     4
Name: count, dtype: int64


## 2. Split por clase

In [37]:
aves_df = df[df['class'] == 'Aves'].copy().reset_index(drop=True)
mam_df  = df[df['class'] == 'Mammalia'].copy().reset_index(drop=True)

print(f'Aves:     {len(aves_df)} spp')
print(f'Mammalia: {len(mam_df)} spp')

Aves:     1478 spp
Mammalia: 226 spp


## 3. Definición de features por clase

In [38]:
# Features comunes a ambas clases
FEATURES_BASE = [
    # Ocurrencias
    'n_occ',
    # Climáticas (WorldClim)
    'bio1',   # temperatura media anual
    'bio4',   # estacionalidad temperatura
    'bio7',   # rango temperatura anual
    'bio12',  # precipitación anual
    'bio14',  # precipitación mes más seco
    'bio15',  # estacionalidad precipitación
    # Deforestación (GFW)
    'pct_forest_loss_total',   # 2001-2023
    'pct_forest_loss_recent',  # 2015-2023
    # Cobertura terrestre (MapBiomas)
    'lc_forest',
    'lc_natural',
    'lc_agriculture',
    'lc_water',
    # Protección (RAISG)
    'pct_anp', 'anp_dist_km',
    'pct_ti',  'ti_dist_km',
    # Presión minera (RAISG)
    'pct_min_ilegal',  'min_ilegal_dist_km',
    'pct_concesion',   'concesion_dist_km',
    # Presión extractiva
    'pct_petroleo', 'petroleo_dist_km',
    # Accesibilidad
    'via_dist_km',
    # Perturbación
    'quemas_freq',
]

# Traits AVONET (solo aves)
FEATURES_AVONET_NUM = [
    'Mass', 'Beak.Length_Culmen', 'Beak.Width', 'Beak.Depth',
    'Tarsus.Length', 'Wing.Length', 'Kipps.Distance',
    'Habitat.Density', 'Migration', 'Range.Size',
]
FEATURES_AVONET_CAT = ['Habitat', 'Trophic.Level', 'Trophic.Niche', 'Primary.Lifestyle']

# Traits PanTHERIA (solo mamíferos)
FEATURES_PANTHERIA = ['body_mass_g', 'trophic_level', 'activity_cycle', 'range_area_km2']

FEATURES_AVES     = FEATURES_BASE + FEATURES_AVONET_NUM + FEATURES_AVONET_CAT
FEATURES_MAMIFEROS = FEATURES_BASE + FEATURES_PANTHERIA

print(f'Features aves:      {len(FEATURES_AVES)}')
print(f'Features mamíferos: {len(FEATURES_MAMIFEROS)}')

Features aves:      39
Features mamíferos: 29


## 4. Limpieza — Aves

In [39]:
# Diagnóstico de nulos en aves
aves_feats = aves_df[FEATURES_AVES + ['species', 'iucn_categoria', 'threatened']].copy()

print('Nulos en features de aves:')
nulls_aves = aves_feats[FEATURES_AVES].isnull().sum()
print(nulls_aves[nulls_aves > 0])
print(f'\nAves sin traits AVONET: {aves_feats[FEATURES_AVONET_NUM[0]].isna().sum()}')

Nulos en features de aves:
bio1                    5
bio4                    5
bio7                    5
bio12                   5
bio14                   5
bio15                   5
Mass                  161
Beak.Length_Culmen    161
Beak.Width            161
Beak.Depth            161
Tarsus.Length         161
Wing.Length           161
Kipps.Distance        161
Habitat.Density       161
Migration             162
Range.Size            161
Habitat               164
Trophic.Level         161
Trophic.Niche         161
Primary.Lifestyle     161
dtype: int64

Aves sin traits AVONET: 161


In [40]:
# Imputación numérica con mediana (calculada sobre filas con IUCN para no filtrar)
# Separar antes de imputar para evitar data leakage del set de predicción
aves_train_raw   = aves_feats[aves_feats['threatened'].notna()].copy()
aves_predict_raw = aves_feats[aves_feats['threatened'].isna()].copy()

print(f'Aves train (con IUCN):   {len(aves_train_raw)}')
print(f'Aves predict (sin IUCN): {len(aves_predict_raw)}')
print(f'\nDistribución target en train:')
print(aves_train_raw['threatened'].value_counts())

Aves train (con IUCN):   1313
Aves predict (sin IUCN): 165

Distribución target en train:
threatened
0.0    1277
1.0      36
Name: count, dtype: int64


In [41]:
# Imputar numéricas con mediana del train set
NUM_AVES = FEATURES_BASE + FEATURES_AVONET_NUM

medians_aves = aves_train_raw[NUM_AVES].median()

aves_train_raw[NUM_AVES]   = aves_train_raw[NUM_AVES].fillna(medians_aves)
aves_predict_raw[NUM_AVES] = aves_predict_raw[NUM_AVES].fillna(medians_aves)

print('Nulos restantes (numéricos):')
print(aves_train_raw[NUM_AVES].isnull().sum()[lambda x: x > 0])
print('(ninguno si está bien)')

Nulos restantes (numéricos):
Series([], dtype: int64)
(ninguno si está bien)


In [42]:
# Imputar categóricas con moda del train set
for col in FEATURES_AVONET_CAT:
    mode_val = aves_train_raw[col].mode()[0]
    aves_train_raw[col]   = aves_train_raw[col].fillna(mode_val)
    aves_predict_raw[col] = aves_predict_raw[col].fillna(mode_val)
    print(f'{col}: mode={mode_val}, nulos train={aves_train_raw[col].isna().sum()}')

Habitat: mode=Forest, nulos train=0
Trophic.Level: mode=Carnivore, nulos train=0
Trophic.Niche: mode=Invertivore, nulos train=0
Primary.Lifestyle: mode=Insessorial, nulos train=0


In [43]:
# One-hot encoding de categóricas
# Fit en train, apply a ambos (mismo set de columnas)

def encode_categoricals(train_df, predict_df, cat_cols):
    """OHE fit en train, transform en ambos. Garantiza mismas columnas."""
    train_enc = pd.get_dummies(train_df, columns=cat_cols, dtype=float)
    predict_enc = pd.get_dummies(predict_df, columns=cat_cols, dtype=float)
    
    # Alinear columnas (predict puede tener categorías faltantes)
    missing_in_pred = set(train_enc.columns) - set(predict_enc.columns)
    for col in missing_in_pred:
        predict_enc[col] = 0.0
    
    # Mismo orden de columnas
    predict_enc = predict_enc[train_enc.columns]
    return train_enc, predict_enc

aves_train_enc, aves_predict_enc = encode_categoricals(
    aves_train_raw, aves_predict_raw, FEATURES_AVONET_CAT
)

print(f'Shape aves train tras OHE:   {aves_train_enc.shape}')
print(f'Shape aves predict tras OHE: {aves_predict_enc.shape}')

# Columnas OHE generadas
ohe_cols = [c for c in aves_train_enc.columns if any(c.startswith(p+'_') for p in FEATURES_AVONET_CAT)]
print(f'Columnas OHE: {len(ohe_cols)}')

Shape aves train tras OHE:   (1313, 67)
Shape aves predict tras OHE: (165, 67)
Columnas OHE: 29


## 5. Limpieza — Mamíferos

In [44]:
mam_feats = mam_df[FEATURES_MAMIFEROS + ['species', 'iucn_categoria', 'threatened']].copy()

print('Nulos en features de mamíferos:')
nulls_mam = mam_feats[FEATURES_MAMIFEROS].isnull().sum()
print(nulls_mam[nulls_mam > 0])

# Split train/predict
mam_train_raw   = mam_feats[mam_feats['threatened'].notna()].copy()
mam_predict_raw = mam_feats[mam_feats['threatened'].isna()].copy()

print(f'\nMamíferos train:   {len(mam_train_raw)}')
print(f'Mamíferos predict: {len(mam_predict_raw)}')
print(f'\nDistribución target en train:')
print(mam_train_raw['threatened'].value_counts())

Nulos en features de mamíferos:
bio1                1
bio4                1
bio7                1
bio12               1
bio14               1
bio15               1
body_mass_g        55
trophic_level      89
activity_cycle    123
range_area_km2     57
dtype: int64

Mamíferos train:   189
Mamíferos predict: 37

Distribución target en train:
threatened
0.0    176
1.0     13
Name: count, dtype: int64


In [45]:
# Imputar PanTHERIA con mediana del train
medians_mam = mam_train_raw[FEATURES_MAMIFEROS].median()

mam_train_raw[FEATURES_MAMIFEROS]   = mam_train_raw[FEATURES_MAMIFEROS].fillna(medians_mam)
mam_predict_raw[FEATURES_MAMIFEROS] = mam_predict_raw[FEATURES_MAMIFEROS].fillna(medians_mam)

print('Nulos restantes en mamíferos:')
remaining = mam_train_raw[FEATURES_MAMIFEROS].isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else 'Ninguno ✓')

Nulos restantes en mamíferos:
Ninguno ✓


## 6. Resumen final

In [46]:
# Columnas finales de features (sin metadata)
META_COLS = ['species', 'iucn_categoria', 'threatened']

feat_cols_aves = [c for c in aves_train_enc.columns if c not in META_COLS]
feat_cols_mam  = [c for c in mam_train_raw.columns if c not in META_COLS]

print('=== AVES ===')
print(f'  Train:   {len(aves_train_enc)} spp x {len(feat_cols_aves)} features')
print(f'  Predict: {len(aves_predict_enc)} spp x {len(feat_cols_aves)} features')
print(f'  Amenazadas (1): {int(aves_train_enc["threatened"].sum())}')
print(f'  No amenazadas (0): {int((aves_train_enc["threatened"]==0).sum())}')
print(f'  Ratio imbalance: 1:{int((aves_train_enc["threatened"]==0).sum() / aves_train_enc["threatened"].sum())}')

print()
print('=== MAMÍFEROS ===')
print(f'  Train:   {len(mam_train_raw)} spp x {len(feat_cols_mam)} features')
print(f'  Predict: {len(mam_predict_raw)} spp x {len(feat_cols_mam)} features')
print(f'  Amenazadas (1): {int(mam_train_raw["threatened"].sum())}')
print(f'  No amenazadas (0): {int((mam_train_raw["threatened"]==0).sum())}')
print(f'  Ratio imbalance: 1:{int((mam_train_raw["threatened"]==0).sum() / mam_train_raw["threatened"].sum())}')

=== AVES ===
  Train:   1313 spp x 64 features
  Predict: 165 spp x 64 features
  Amenazadas (1): 36
  No amenazadas (0): 1277
  Ratio imbalance: 1:35

=== MAMÍFEROS ===
  Train:   189 spp x 29 features
  Predict: 37 spp x 29 features
  Amenazadas (1): 13
  No amenazadas (0): 176
  Ratio imbalance: 1:13


In [47]:
print('Features finales — AVES:')
for c in feat_cols_aves:
    print(f'  {c}')

Features finales — AVES:
  n_occ
  bio1
  bio4
  bio7
  bio12
  bio14
  bio15
  pct_forest_loss_total
  pct_forest_loss_recent
  lc_forest
  lc_natural
  lc_agriculture
  lc_water
  pct_anp
  anp_dist_km
  pct_ti
  ti_dist_km
  pct_min_ilegal
  min_ilegal_dist_km
  pct_concesion
  concesion_dist_km
  pct_petroleo
  petroleo_dist_km
  via_dist_km
  quemas_freq
  Mass
  Beak.Length_Culmen
  Beak.Width
  Beak.Depth
  Tarsus.Length
  Wing.Length
  Kipps.Distance
  Habitat.Density
  Migration
  Range.Size
  Habitat_Coastal
  Habitat_Forest
  Habitat_Grassland
  Habitat_Human Modified
  Habitat_Marine
  Habitat_Riverine
  Habitat_Rock
  Habitat_Shrubland
  Habitat_Wetland
  Habitat_Woodland
  Trophic.Level_Carnivore
  Trophic.Level_Herbivore
  Trophic.Level_Omnivore
  Trophic.Level_Scavenger
  Trophic.Niche_Aquatic predator
  Trophic.Niche_Frugivore
  Trophic.Niche_Granivore
  Trophic.Niche_Herbivore aquatic
  Trophic.Niche_Herbivore terrestrial
  Trophic.Niche_Invertivore
  Trophic.Niche_Ne

In [48]:
print('Features finales — MAMÍFEROS:')
for c in feat_cols_mam:
    print(f'  {c}')

Features finales — MAMÍFEROS:
  n_occ
  bio1
  bio4
  bio7
  bio12
  bio14
  bio15
  pct_forest_loss_total
  pct_forest_loss_recent
  lc_forest
  lc_natural
  lc_agriculture
  lc_water
  pct_anp
  anp_dist_km
  pct_ti
  ti_dist_km
  pct_min_ilegal
  min_ilegal_dist_km
  pct_concesion
  concesion_dist_km
  pct_petroleo
  petroleo_dist_km
  via_dist_km
  quemas_freq
  body_mass_g
  trophic_level
  activity_cycle
  range_area_km2


## 7. Guardado

In [49]:
aves_train_enc.to_parquet(DATA / 'aves_train.parquet', engine='fastparquet', index=False)
aves_predict_enc.to_parquet(DATA / 'aves_predict.parquet', engine='fastparquet', index=False)
mam_train_raw.to_parquet(DATA / 'mamiferos_train.parquet', engine='fastparquet', index=False)
mam_predict_raw.to_parquet(DATA / 'mamiferos_predict.parquet', engine='fastparquet', index=False)

print('Guardados:')
for f in ['aves_train.parquet', 'aves_predict.parquet', 'mamiferos_train.parquet', 'mamiferos_predict.parquet']:
    p = DATA / f
    print(f'  {f}: {p.stat().st_size / 1024:.1f} KB')

Guardados:
  aves_train.parquet: 258.2 KB
  aves_predict.parquet: 42.8 KB
  mamiferos_train.parquet: 28.3 KB
  mamiferos_predict.parquet: 11.8 KB
